# ClimateClaimVerifier — Colab Shakeout (Week 7 multimodal)

Runs the **capped shakeout** end-to-end on Colab **GPU**: fresh ingest (with the new
image/reshare/profile capture) -> classify -> **gated `qwen2.5vl` vision on edge cases** ->
export the DB to Google Drive. You then pull that DB locally (Phase 3) to evaluate + view the dashboard.

**Before running:** Runtime -> Change runtime type -> **GPU (T4)**. This tests on branch
`multimodal-edge-gating` only; `main` is untouched.

## 1. Confirm GPU

In [ ]:
!nvidia-smi

## 2. Clone repo + checkout the test branch (public repo, no PAT needed)

In [ ]:
!git clone https://github.com/Sadaf-Burhan/ClimateClaimVerifier.git
%cd ClimateClaimVerifier
!git checkout multimodal-edge-gating
!git log --oneline -4

## 3. Install the package + dependencies

In [ ]:
# Installs climate_verifier + its deps (chromadb, ollama, atproto, sentence-transformers, ...).
!pip install -q -e .

## 4. Install + start Ollama, pull the models

`qwen2.5:3b` is the classifier; `qwen2.5vl:7b` is the vision model (must match
`config.yaml -> vision.model`). If the vision tag 404s, try `qwen2.5vl:3b` and update the
config value to match.

In [ ]:
!curl -fsSL https://ollama.com/install.sh | sh
import subprocess, time
subprocess.Popen(["ollama", "serve"])
time.sleep(6)
!ollama pull qwen2.5:3b
!ollama pull qwen2.5vl:7b
!ollama list

## 5. Bluesky credentials

Entered via `getpass` so nothing is stored in the notebook. **Use the rotated app password.**
`load_dotenv()` does not override these, so setting them here is enough.

In [ ]:
import os, getpass
os.environ["BLUESKY_HANDLE"] = input("Bluesky handle (e.g. name.bsky.social): ").strip()
os.environ["BLUESKY_APP_PASSWORD"] = getpass.getpass("Bluesky app password: ").strip()
print("creds set for", os.environ["BLUESKY_HANDLE"])

## 6. Capped shakeout — set `bluesky_limit: 15`

A regex patch (preserves the config comments). For the FULL run later, set it back to 100.

In [ ]:
import re, pathlib
p = pathlib.Path("src/climate_verifier/config.yaml")
p.write_text(re.sub(r"(bluesky_limit:\s*)\d+", r"\g<1>15", p.read_text()))
print([l for l in p.read_text().splitlines() if "bluesky_limit" in l])

## 7. Ingest (Bluesky + GDELT, topic-filtered)

Call `run_ingestion_cycle()` directly (the module's `__main__` starts a *blocking* scheduler —
don't use `!python -m ...scheduler` here). This captures the new fields: `has_image`, `image_url`,
`image_alt`, `reshare_of_*`, `external_*`, `author_bio/post_count/created_at`. GDELT is
rate-limited (~10s/keyword) so this is the slow cell — a few minutes even at the shakeout cap.

In [ ]:
from climate_verifier.ingestion.scheduler import run_ingestion_cycle
run_ingestion_cycle()   # fresh DB (no prior run -> guard passes)

## 8. Classify (claim / opinion, single-post for recall)

In [ ]:
!python -m climate_verifier.pipeline.claim_classifier

## 9. Edge-case vision

First a **dry run** — which posts *would* escalate (claim + image + weak category, not resolved by
metadata). Then the real **`qwen2.5vl`** pass writes `vision_signal` for those.

In [ ]:
!python -m climate_verifier.pipeline.vision            # dry run: list edge cases

In [ ]:
!python -m climate_verifier.pipeline.vision --gate-and-analyze   # run vision

## 10. Sanity check — captured fields + vision signals

In [ ]:
import sqlite3, json
con = sqlite3.connect("data/ingested.db"); con.row_factory = sqlite3.Row
def n(q): return con.execute(q).fetchone()[0]
print("posts total        :", n("SELECT COUNT(*) FROM posts"))
print("  bluesky w/ image :", n("SELECT COUNT(*) FROM posts WHERE source='bluesky' AND has_image=1"))
print("  with external url:", n("SELECT COUNT(*) FROM posts WHERE external_url IS NOT NULL"))
print("  quote reshares   :", n("SELECT COUNT(*) FROM posts WHERE reshare_of_author IS NOT NULL"))
print("  with author bio  :", n("SELECT COUNT(*) FROM posts WHERE author_bio IS NOT NULL AND author_bio<>''"))
print("classified         :", n("SELECT COUNT(*) FROM classifications"))
print("vision_signal set  :", n("SELECT COUNT(*) FROM posts WHERE vision_signal IS NOT NULL"))
print("\n--- vision signals ---")
for r in con.execute("SELECT author, text, vision_signal FROM posts WHERE vision_signal IS NOT NULL LIMIT 10"):
    vs = json.loads(r["vision_signal"])
    print(f"@{r['author'][:20]:20} type={vs.get('image_type'):16} depicts={vs.get('depicts_claim'):9} | {r['text'][:55]}")
con.close()

## 11. Export the DB to Google Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
import os, shutil
dest = "/content/drive/MyDrive/ClimateClaimVerifier"
os.makedirs(dest, exist_ok=True)
shutil.copy("data/ingested.db", f"{dest}/ingested_shakeout.db")
print("Exported ->", f"{dest}/ingested_shakeout.db")

## Next: Phase 3 (local CPU, VS Code)

1. Download `ingested_shakeout.db` from Drive -> save as `data/ingested.db`
   (your `data/ingested_backup_premultimodal.db` still holds the original).
2. Re-flag eval/train against the frozen CSV, then:
   ```
   uv run python -m climate_verifier.pipeline.evaluate      # recall/precision vs baseline
   uv run python -m climate_verifier.pipeline.evidence --build
   uv run streamlit run app.py                              # spot-check edge cases + vision signal
   ```
3. If edge-case precision improves with recall >= 0.90 -> merge `multimodal-edge-gating` -> `main`.

**For the full run:** rerun this notebook with `bluesky_limit` back at 100 (Cell 6).